# Лабораторная работа №5

## Обучение на основе временных различий

**Цель лабораторной работы:** ознакомление с базовыми методами обучения с подкреплением на основе временных различий.

**Задание:**

На основе рассмотренного на лекции примера необходимо реализовать следующие алгоритмы:

1. SARSA.
2. Q-обучение.
3. Двойное Q-обучение.

Для выполнения работы используется среда **Blackjack-v1** из библиотеки **Gymnasium**.

Среда **Toy Text / Frozen Lake** не используется.


## 1. Установка и импорт библиотек

В данном блоке подключаются библиотеки для работы со средой обучения с подкреплением, численных вычислений, построения графиков и хранения результатов.

Если библиотека `gymnasium` не установлена, ее можно установить командой:

```python
!pip install gymnasium
```


In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("gymnasium") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gymnasium"])

import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)

## 2. Описание среды Blackjack-v1

В лабораторной работе используется среда **Blackjack-v1**.

Задача агента — принимать решения в игре Blackjack:

- действие `0` — остановиться;
- действие `1` — взять еще одну карту.

Состояние среды задается кортежем:

```python
(player_sum, dealer_card, usable_ace)
```

где:

- `player_sum` — сумма карт игрока;
- `dealer_card` — открытая карта дилера;
- `usable_ace` — наличие туза, который можно считать как 11 без перебора.

Цель агента — максимизировать суммарное вознаграждение.


In [ ]:
env = gym.make("Blackjack-v1", sab=True)

print("Среда:", env)
print("Пространство действий:", env.action_space)
print("Пример состояния после reset:")

state, info = env.reset(seed=42)
print(state)

## 3. Вспомогательные функции

В этом блоке реализуются общие функции, которые используются всеми алгоритмами:

1. `epsilon_greedy_policy` — выбор действия по ε-жадной стратегии.
2. `moving_average` — сглаживание графиков награды.
3. `evaluate_policy` — оценка обученной стратегии без случайных действий.

ε-жадная стратегия нужна для баланса между исследованием среды и использованием уже найденных хороших действий.


In [ ]:
def epsilon_greedy_policy(Q, state, n_actions, epsilon):
    if np.random.random() < epsilon:
        return np.random.randint(n_actions)
    return int(np.argmax(Q[state]))


def moving_average(values, window=500):
    values = np.array(values)
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")


def evaluate_policy(Q, episodes=5000, seed=123):
    eval_env = gym.make("Blackjack-v1", sab=True)
    rewards = []

    for episode in range(episodes):
        state, info = eval_env.reset(seed=seed + episode)
        done = False
        total_reward = 0

        while not done:
            action = int(np.argmax(Q[state]))
            next_state, reward, terminated, truncated, info = eval_env.step(action)
            done = terminated or truncated
            total_reward += reward
            state = next_state

        rewards.append(total_reward)

    eval_env.close()

    return {
        "Средняя награда": np.mean(rewards),
        "Доля побед": np.mean(np.array(rewards) > 0),
        "Доля ничьих": np.mean(np.array(rewards) == 0),
        "Доля поражений": np.mean(np.array(rewards) < 0)
    }

## 4. Алгоритм SARSA

**SARSA** — это on-policy алгоритм обучения с подкреплением.

Название SARSA связано с последовательностью:

```text
State → Action → Reward → State → Action
```

Формула обновления:

$$
Q(s,a) = Q(s,a) + \alpha [r + \gamma Q(s',a') - Q(s,a)]
$$

Особенность SARSA состоит в том, что для обновления используется следующее действие, выбранное текущей ε-жадной политикой.


In [ ]:
def train_sarsa(
    episodes=100000,
    alpha=0.1,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.99995
):
    env = gym.make("Blackjack-v1", sab=True)
    n_actions = env.action_space.n

    Q = defaultdict(lambda: np.zeros(n_actions))
    rewards = []
    epsilon = epsilon_start

    for episode in range(episodes):
        state, info = env.reset()
        action = epsilon_greedy_policy(Q, state, n_actions, epsilon)

        done = False
        total_reward = 0

        while not done:
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            if done:
                td_target = reward
            else:
                next_action = epsilon_greedy_policy(Q, next_state, n_actions, epsilon)
                td_target = reward + gamma * Q[next_state][next_action]

            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error

            total_reward += reward

            if not done:
                state = next_state
                action = next_action

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards.append(total_reward)

    env.close()
    return Q, rewards


Q_sarsa, rewards_sarsa = train_sarsa()

print("SARSA обучен.")
print("Количество эпизодов:", len(rewards_sarsa))
print("Средняя награда за последние 1000 эпизодов:", np.mean(rewards_sarsa[-1000:]))

## 5. Алгоритм Q-обучения

**Q-обучение** — это off-policy алгоритм.

Он обучает оптимальную стратегию независимо от того, какие действия фактически выбирались во время исследования среды.

Формула обновления:

$$
Q(s,a) = Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)]
$$

Главное отличие от SARSA: в целевом значении используется максимум по действиям в следующем состоянии.


In [ ]:
def train_q_learning(
    episodes=100000,
    alpha=0.1,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.99995
):
    env = gym.make("Blackjack-v1", sab=True)
    n_actions = env.action_space.n

    Q = defaultdict(lambda: np.zeros(n_actions))
    rewards = []
    epsilon = epsilon_start

    for episode in range(episodes):
        state, info = env.reset()
        done = False
        total_reward = 0

        while not done:
            action = epsilon_greedy_policy(Q, state, n_actions, epsilon)

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            if done:
                td_target = reward
            else:
                td_target = reward + gamma * np.max(Q[next_state])

            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error

            total_reward += reward
            state = next_state

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards.append(total_reward)

    env.close()
    return Q, rewards


Q_q_learning, rewards_q_learning = train_q_learning()

print("Q-обучение выполнено.")
print("Количество эпизодов:", len(rewards_q_learning))
print("Средняя награда за последние 1000 эпизодов:", np.mean(rewards_q_learning[-1000:]))

## 6. Алгоритм Double Q-learning

**Двойное Q-обучение** используется для уменьшения переоценки значений действий.

В обычном Q-learning используется максимум:

$$
\max_{a'} Q(s',a')
$$

Из-за этого значения действий могут быть завышены.

В Double Q-learning используются две таблицы:

$$
Q_1
$$

и

$$
Q_2
$$

На каждой итерации случайно обновляется одна из них. При этом выбор действия выполняется по одной таблице, а оценка — по другой.


In [ ]:
def train_double_q_learning(
    episodes=100000,
    alpha=0.1,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.99995
):
    env = gym.make("Blackjack-v1", sab=True)
    n_actions = env.action_space.n

    Q1 = defaultdict(lambda: np.zeros(n_actions))
    Q2 = defaultdict(lambda: np.zeros(n_actions))

    rewards = []
    epsilon = epsilon_start

    for episode in range(episodes):
        state, info = env.reset()
        done = False
        total_reward = 0

        while not done:
            Q_sum = defaultdict(lambda: np.zeros(n_actions))
            for key in set(list(Q1.keys()) + list(Q2.keys())):
                Q_sum[key] = Q1[key] + Q2[key]

            action = epsilon_greedy_policy(Q_sum, state, n_actions, epsilon)

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            if np.random.random() < 0.5:
                if done:
                    td_target = reward
                else:
                    best_next_action = int(np.argmax(Q1[next_state]))
                    td_target = reward + gamma * Q2[next_state][best_next_action]

                td_error = td_target - Q1[state][action]
                Q1[state][action] += alpha * td_error
            else:
                if done:
                    td_target = reward
                else:
                    best_next_action = int(np.argmax(Q2[next_state]))
                    td_target = reward + gamma * Q1[next_state][best_next_action]

                td_error = td_target - Q2[state][action]
                Q2[state][action] += alpha * td_error

            total_reward += reward
            state = next_state

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards.append(total_reward)

    env.close()

    Q_final = defaultdict(lambda: np.zeros(n_actions))
    for key in set(list(Q1.keys()) + list(Q2.keys())):
        Q_final[key] = Q1[key] + Q2[key]

    return Q_final, rewards


Q_double, rewards_double = train_double_q_learning()

print("Double Q-learning выполнен.")
print("Количество эпизодов:", len(rewards_double))
print("Средняя награда за последние 1000 эпизодов:", np.mean(rewards_double[-1000:]))

## 7. Сравнение обучения по награде

В данном блоке строится график скользящего среднего вознаграждения.

Скользящее среднее позволяет увидеть общую динамику обучения без сильных случайных колебаний отдельных эпизодов.


In [ ]:
window = 1000

plt.figure(figsize=(12, 6))

plt.plot(moving_average(rewards_sarsa, window), label="SARSA")
plt.plot(moving_average(rewards_q_learning, window), label="Q-learning")
plt.plot(moving_average(rewards_double, window), label="Double Q-learning")

plt.xlabel("Эпизод")
plt.ylabel("Скользящее среднее награды")
plt.title("Сравнение алгоритмов обучения на основе временных различий")
plt.legend()
plt.grid(True)
plt.show()

## 8. Оценка обученных стратегий

После обучения необходимо проверить качество полученных стратегий.

Для оценки используется отдельный набор эпизодов, где агент действует жадно, то есть выбирает действие с максимальным Q-значением без случайного исследования.


In [ ]:
eval_sarsa = evaluate_policy(Q_sarsa)
eval_q_learning = evaluate_policy(Q_q_learning)
eval_double = evaluate_policy(Q_double)

results_df = pd.DataFrame([
    {"Алгоритм": "SARSA", **eval_sarsa},
    {"Алгоритм": "Q-learning", **eval_q_learning},
    {"Алгоритм": "Double Q-learning", **eval_double}
])

display(results_df)

## 9. График сравнения средней награды

На данном графике сравнивается средняя награда обученных стратегий.

Чем выше средняя награда, тем лучше стратегия агента.

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(results_df["Алгоритм"], results_df["Средняя награда"])
plt.ylabel("Средняя награда")
plt.title("Сравнение средней награды после обучения")
plt.grid(axis="y")
plt.show()

## 10. График доли побед

В Blackjack важным показателем является доля побед агента.

На графике сравнивается, как часто агент выигрывает после обучения разными алгоритмами.

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(results_df["Алгоритм"], results_df["Доля побед"])
plt.ylabel("Доля побед")
plt.title("Сравнение доли побед после обучения")
plt.grid(axis="y")
plt.show()

## 11. Пример работы обученного агента

В данном блоке демонстрируется один эпизод игры Blackjack после обучения.

Для примера используется стратегия, полученная методом Q-learning.

In [ ]:
demo_env = gym.make("Blackjack-v1", sab=True, render_mode=None)

state, info = demo_env.reset(seed=7)
done = False
total_reward = 0
step = 1

print("Начальное состояние:", state)

while not done:
    action = int(np.argmax(Q_q_learning[state]))
    next_state, reward, terminated, truncated, info = demo_env.step(action)
    done = terminated or truncated

    print(f"Шаг {step}:")
    print("  Состояние:", state)
    print("  Действие:", action, "(0 — остановиться, 1 — взять карту)")
    print("  Следующее состояние:", next_state)
    print("  Награда:", reward)

    total_reward += reward
    state = next_state
    step += 1

demo_env.close()

print("Итоговая награда за эпизод:", total_reward)

## 12. Вывод

В ходе лабораторной работы были реализованы три алгоритма обучения с подкреплением на основе временных различий:

1. SARSA.
2. Q-обучение.
3. Двойное Q-обучение.

Для эксперимента использовалась среда **Blackjack-v1** из библиотеки Gymnasium.

Алгоритм SARSA является on-policy методом, так как обновляет Q-значения с учетом действия, выбранного текущей политикой.

Q-learning является off-policy методом, так как при обновлении использует максимальное значение Q-функции в следующем состоянии.

Double Q-learning использует две Q-таблицы и позволяет уменьшить эффект переоценки значений действий.

По результатам эксперимента были построены графики обучения и выполнено сравнение стратегий по средней награде и доле побед.


In [ ]:
print("Лабораторная работа №5 выполнена.")
print("Реализованы алгоритмы SARSA, Q-learning и Double Q-learning.")
print("Среда: Blackjack-v1.")
print("Frozen Lake не использовалась.")